# Tratando os dados dp PgAdmin

- @author: Guilherme Nogueira

## Importações

In [ ]:
# =========================================================
# BIBLIOTECAS
# =========================================================
import os
import re
import unicodedata
import pandas as pd
import numpy as np
from datetime import datetime
from sqlalchemy import text
from dotenv import load_dotenv
from sqlalchemy import URL, create_engine, text
from sqlalchemy.exc import SQLAlchemyError

from sqlalchemy import text


# =========================================================
# FUNÇÕES
# =========================================================
import sys
from pathlib import Path

PASTA_FUNCOES = Path(r"C:\Users\Guilherme\LABOR RURAL\Analytics - Departamento Analytics\TEMP\GUILHERME\SCRIPTS\projetcs\lr-functions\functions")

if not PASTA_FUNCOES.exists():
    raise FileNotFoundError(
        f"Pasta das funções não encontrada:\n{PASTA_FUNCOES}"
    )

if str(PASTA_FUNCOES) not in sys.path:
    sys.path.insert(0, str(PASTA_FUNCOES))

from excel_format import (
    exportar_xlsx_formatado,
    exportar_varias_abas_xlsx,
)

print("Funções de Excel importadas com sucesso.")

Funções de Excel importadas com sucesso.


### Configurações de conexão

In [2]:
# Carrega as variáveis do arquivo .env
load_dotenv()

# Configurações de conexão
PG_HOST = os.getenv("PG_HOST")
PG_PORT = int(os.getenv("PG_PORT", "5432"))
PG_DATABASE = os.getenv("PG_DATABASE")
PG_USER = os.getenv("PG_USER")
PG_PASSWORD = os.getenv("PG_PASSWORD")
PG_SCHEMA = os.getenv("PG_SCHEMA", "analytics_mart")

# Verifica se as configurações obrigatórias foram preenchidas
configuracoes = {
    "PG_HOST": PG_HOST,
    "PG_DATABASE": PG_DATABASE,
    "PG_USER": PG_USER,
    "PG_PASSWORD": PG_PASSWORD,
}

faltantes = [
    nome
    for nome, valor in configuracoes.items()
    if valor is None or str(valor).strip() == ""
]

if faltantes:
    raise ValueError(
        "As seguintes variáveis não foram preenchidas no arquivo .env: "
        + ", ".join(faltantes)
    )


# Criação segura da URL de conexão
url_conexao = URL.create(
    drivername="postgresql+psycopg",
    username=PG_USER,
    password=PG_PASSWORD,
    host=PG_HOST,
    port=PG_PORT,
    database=PG_DATABASE,
)


# Engine de conexão
engine = create_engine(
    url_conexao,
    pool_pre_ping=True,
)

### Testar a conexão

In [ ]:
try:
    with engine.connect() as conexao:
        resultado = conexao.execute(
            text(
                """
                SELECT
                    current_database() AS banco,
                    current_user AS usuario,
                    current_schema() AS schema_atual,
                    version() AS versao
                """
            )
        ).mappings().one()
    
    print("Conexão realizada com sucesso!")
    print(f"Banco: {resultado['banco']}")
    print(f"Usuário: {resultado['usuario']}")
    print(f"Schema atual: {resultado['schema_atual']}")

except SQLAlchemyError as erro:
    print("Não foi possível conectar ao PostgreSQL.")
    raise erro

Conexão realizada com sucesso!
Banco: postgres
Usuário: lradmin
Schema atual: public


### Configuração das Views

In [ ]:
# ============================================================
# CONFIGURAÇÕES DAS VIEWS
# ============================================================

# Chaves usadas nos merges
CHAVES = ["id_property", "reference_month"]

# Período analisado
DATA_INICIAL = pd.Timestamp("2025-01-01")

# Primeiro dia do mês atual
DATA_FINAL = (pd.Timestamp.today().to_period("M").to_timestamp())

# View principal da tabela final
VIEW_BASE = "vw_revenue"

# Views que serão adicionadas à view principal
VIEWS_MERGE = [
    "vw_cattle",
    # "vw_culture_expense",
    "vw_expense",
    "vw_feeding",
    "vw_labor",
    "vw_own_milk",
    # "vw_production",
    "vw_area_ativa_mes",
    "vw_dairy_production_system_monthly",
    "vw_asset_payment_history_monthly",
]

# Lista completa de views autorizadas para importação
views = list(
    dict.fromkeys(
        [VIEW_BASE] + VIEWS_MERGE
    )
)

# ============================================================
# CONFIGURAÇÃO ESPECÍFICA DA ÁREA ATIVA
# ============================================================

NOME_VIEW_AREA = "vw_area_ativa_mes"

COLUNAS_AREA_ATIVA = [
    "month_start",
    "id_property",
    "hectares_owned_mes",
    "hectares_rented_mes",
    "hectares_total_mes",
    "raw_land_value_avg_weighted_mes",
]

# ============================================================
# FUNÇÃO DE IMPORTAÇÃO
# ============================================================

def importar_view(nome_view: str, engine, schema: str, ordenar_por: str | None = None,) -> pd.DataFrame:
    """
    Importa uma view PostgreSQL para um DataFrame.

    A view precisa estar cadastrada na lista `views`. 
    A ordenação é feita no pandas somente quando a coluna informada existir.
    """

    if nome_view not in views:
        raise ValueError(
            f"View não autorizada: {nome_view}"
        )

    consulta = text(f''' SELECT * FROM "{schema}"."{nome_view}"; ''')

    df = pd.read_sql_query(sql=consulta, con=engine)

    if (ordenar_por is not None and ordenar_por in df.columns):
        df = (df.sort_values(ordenar_por).reset_index(drop=True) )

    return df


# ============================================================
# CONFERÊNCIA DAS CONFIGURAÇÕES
# ============================================================

print(f"View principal: {VIEW_BASE}")

print("\nViews usadas nos merges:")
for nome_view in VIEWS_MERGE:
    print(f"- {nome_view}")

print("\nViews autorizadas para importação:")
for nome_view in views:
    print(f"- {nome_view}")

print(f"\nPeríodo: " f"{DATA_INICIAL:%Y-%m} a {DATA_FINAL:%Y-%m}" )

View principal: vw_revenue

Views usadas nos merges:
- vw_asset_payment_history_monthly
- vw_cattle
- vw_expense
- vw_feeding
- vw_labor
- vw_own_milk
- vw_area_ativa_mes
- vw_dairy_production_system_monthly

Views autorizadas para importação:
- vw_revenue
- vw_asset_payment_history_monthly
- vw_cattle
- vw_expense
- vw_feeding
- vw_labor
- vw_own_milk
- vw_area_ativa_mes
- vw_dairy_production_system_monthly

Período: 2025-01 a 2026-07


## Importando as views do PdAdmin

In [ ]:
consulta_conexao = text("""
    SELECT
        current_database() AS banco_atual,
        current_user AS usuario_atual,
        current_schema() AS schema_atual,
        current_setting('search_path') AS search_path,
        inet_server_addr() AS endereco_servidor,
        inet_server_port() AS porta_servidor;
""")

with engine.connect() as conexao:
    diagnostico_conexao = pd.read_sql_query(consulta_conexao, conexao)

display(diagnostico_conexao)

,banco_atual,usuario_atual,schema_atual,search_path,endereco_servidor,porta_servidor
0,postgres,lradmin,public,"""$user"", public",10.34.0.4,5432


#### Importação em massa

In [ ]:
dados_views = {}
resumo_importacao = []

for nome_view in views:
    try:
        print(f"\nImportando {PG_SCHEMA}.{nome_view}...")

        df = importar_view(nome_view=nome_view, engine=engine, schema=PG_SCHEMA, ordenar_por="id_property")

        dados_views[nome_view] = df
        
        resumo_importacao.append({
                "view": nome_view,
                "status": "Importada",
                "linhas": df.shape[0],
                "colunas": df.shape[1],
                "erro": None,
        })

        print(f"Concluído: {df.shape[0]:,} linhas " f"e {df.shape[1]:,} colunas.")

    except Exception as erro:
        dados_views[nome_view] = None
        
        resumo_importacao.append({
                "view": nome_view,
                "status": "Erro",
                "linhas": None,
                "colunas": None,
                "erro": str(erro),
        })
        print(f"Erro ao importar {nome_view}: {erro}")


Importando analytics_mart.vw_revenue...
Concluído: 15,488 linhas e 19 colunas.

Importando analytics_mart.vw_asset_payment_history_monthly...
Concluído: 18,111 linhas e 6 colunas.

Importando analytics_mart.vw_cattle...
Concluído: 15,564 linhas e 18 colunas.

Importando analytics_mart.vw_expense...
Concluído: 15,710 linhas e 21 colunas.

Importando analytics_mart.vw_feeding...
Concluído: 14,759 linhas e 12 colunas.

Importando analytics_mart.vw_labor...
Concluído: 15,457 linhas e 6 colunas.

Importando analytics_mart.vw_own_milk...
Concluído: 12,693 linhas e 11 colunas.

Importando analytics_mart.vw_area_ativa_mes...
Concluído: 35,094 linhas e 14 colunas.

Importando analytics_mart.vw_dairy_production_system_monthly...
Concluído: 30,342 linhas e 3 colunas.


In [ ]:
CHAVES = ["id_property", "reference_month"]

resumo_duplicidades = []
exemplos_duplicidades = {}

for nome_view, df_original in dados_views.items():

    print(f"Verificando {nome_view}...")

    # Trabalhar com uma cópia para não alterar o dado bruto
    df = df_original.copy()

    # ========================================================
    # TRATAMENTO DA VIEW DE ALIMENTAÇÃO
    # ========================================================
    if nome_view == "vw_feeding":

        # A coluna unit não será utilizada
        df = df.drop(
            columns=["unit"],
            errors="ignore",
        )
    
    # ========================================================
    # CONFERIR SE AS CHAVES EXISTEM
    # ========================================================

    colunas_ausentes = [
        coluna
        for coluna in CHAVES
        if coluna not in df.columns
    ]

    if colunas_ausentes:

        resumo_duplicidades.append({
            "view": nome_view,
            "linhas_totais": len(df),
            "linhas_em_chaves_duplicadas": None,
            "chaves_duplicadas": None,
            "chave_unica": False,
            "status": f"Chaves ausentes: {colunas_ausentes}",
        })

        print(
            f"Não foi possível verificar {nome_view}. "
            f"Colunas ausentes: {colunas_ausentes}"
        )

        continue

    # ========================================================
    # PADRONIZAR AS CHAVES
    # ========================================================

    df["id_property"] = (
        df["id_property"]
        .astype("string")
        .str.strip()
    )

    df["reference_month"] = (
        pd.to_datetime(
            df["reference_month"],
            errors="coerce",
        )
        .dt.to_period("M")
        .dt.to_timestamp()
    )

    # ========================================================
    # VERIFICAR DUPLICIDADES
    # ========================================================

    mascara_duplicada = df.duplicated(
        subset=CHAVES,
        keep=False,
    )

    df_duplicados = (
        df.loc[mascara_duplicada]
        .sort_values(CHAVES)
        .copy()
    )

    quantidade_linhas_duplicadas = len(
        df_duplicados
    )

    quantidade_chaves_duplicadas = (
        df_duplicados[CHAVES]
        .drop_duplicates()
        .shape[0]
    )

    resumo_duplicidades.append({
        "view": nome_view,
        "linhas_totais": len(df),
        "linhas_em_chaves_duplicadas": (
            quantidade_linhas_duplicadas
        ),
        "chaves_duplicadas": (
            quantidade_chaves_duplicadas
        ),
        "chave_unica": (
            quantidade_linhas_duplicadas == 0
        ),
        "status": (
            "OK"
            if quantidade_linhas_duplicadas == 0
            else "Possui duplicidades"
        ),
    })

    if quantidade_linhas_duplicadas > 0:

        exemplos_duplicidades[nome_view] = (
            df_duplicados.head(20)
        )

        print(
            f"{nome_view}: "
            f"{quantidade_chaves_duplicadas:,} "
            "chaves duplicadas."
        )

    else:

        print(
            f"{nome_view}: nenhuma duplicidade."
        )


# ============================================================
# RESULTADO
# ============================================================

df_resumo_duplicidades = (
    pd.DataFrame(resumo_duplicidades)
    .sort_values(
        by=["chave_unica", "view"],
        ascending=[True, True],
    )
    .reset_index(drop=True)
)

display(df_resumo_duplicidades)

Verificando vw_revenue...
vw_revenue: nenhuma duplicidade.
Verificando vw_asset_payment_history_monthly...
vw_asset_payment_history_monthly: nenhuma duplicidade.
Verificando vw_cattle...
vw_cattle: nenhuma duplicidade.
Verificando vw_expense...
vw_expense: nenhuma duplicidade.
Verificando vw_feeding...
vw_feeding: nenhuma duplicidade.
Verificando vw_labor...
vw_labor: nenhuma duplicidade.
Verificando vw_own_milk...
vw_own_milk: nenhuma duplicidade.
Verificando vw_area_ativa_mes...
Não foi possível verificar vw_area_ativa_mes. Colunas ausentes: ['reference_month']
Verificando vw_dairy_production_system_monthly...
vw_dairy_production_system_monthly: nenhuma duplicidade.


,view,linhas_totais,linhas_em_chaves_duplicadas,chaves_duplicadas,chave_unica,status
0,vw_area_ativa_mes,35094,NaN,NaN,False,Chaves ausentes: ['reference_month']
1,vw_asset_payment_history_monthly,18111,0.0,0.0,True,OK
2,vw_cattle,15564,0.0,0.0,True,OK
3,vw_dairy_production_system_monthly,30342,0.0,0.0,True,OK
4,vw_expense,15710,0.0,0.0,True,OK
5,vw_feeding,14759,0.0,0.0,True,OK
6,vw_labor,15457,0.0,0.0,True,OK
7,vw_own_milk,12693,0.0,0.0,True,OK
8,vw_revenue,15488,0.0,0.0,True,OK


#### Tratando os dados antes de exportar

##### Funções de tratamento das views

In [ ]:
def preparar_view(df: pd.DataFrame, nome_view: str) -> pd.DataFrame:
    """
    Prepara uma view mensal para os merges.

    - Padroniza id_property;
    - Padroniza o mês de referência;
    - Trata a view de área ativa;
    - Remove unit da view de alimentação;
    - Remove registros sem chave;
    - Filtra o período;
    - Ordena o resultado.
    """

    if df is None:
        raise ValueError( f"A view {nome_view} não foi importada." )

    # Trabalhar com uma cópia para preservar dados_views
    df = df.copy()

    # ========================================================
    # TRATAMENTO ESPECÍFICO DA ÁREA ATIVA
    # ========================================================
    if nome_view == "vw_area_ativa_mes":
        # Padronizar o nome da coluna mensal
        df = df.rename( columns={ "month_start": "reference_month"} )

 
    # ========================================================
    # TRATAMENTO ESPECÍFICO DA ALIMENTAÇÃO
    # ========================================================
    if nome_view == "vw_feeding":
        df = df.drop( columns=["unit"], errors="ignore")

    # ========================================================
    # CONFERIR AS CHAVES
    # ========================================================
    colunas_ausentes = [
        coluna
        for coluna in CHAVES
        if coluna not in df.columns
    ]

    if colunas_ausentes:
        raise KeyError(
            f"A view {nome_view} não possui as colunas "
            f"{colunas_ausentes}."
        )

    # ========================================================
    # PADRONIZAR ID_PROPERTY
    # ========================================================

    df["id_property"] = (
        df["id_property"]
        .astype("string")
        .str.strip()
    )

    # Transformar texto vazio em ausente
    df["id_property"] = df["id_property"].replace( "", pd.NA)

    # ========================================================
    # PADRONIZAR REFERENCE_MONTH
    # ========================================================

    df["reference_month"] = (
        pd.to_datetime(
            df["reference_month"],
            errors="coerce",
        )
        .dt.to_period("M")
        .dt.to_timestamp()
    )

    # ========================================================
    # REMOVER LINHAS SEM CHAVE
    # ========================================================

    registros_sem_chave = ( df[CHAVES] .isna() .any(axis=1) )

    quantidade_sem_chave = int( registros_sem_chave.sum() )

    if quantidade_sem_chave > 0:

        print(
            f"{nome_view}: removendo "
            f"{quantidade_sem_chave:,} linhas sem chave."
        )

        df = df.loc[
            ~registros_sem_chave
        ].copy()

    # ========================================================
    # FILTRAR O PERÍODO
    # ========================================================

    df = df.loc[
        df["reference_month"].between(
            DATA_INICIAL,
            DATA_FINAL,
            inclusive="both",
        )
    ].copy()

    # ========================================================
    # ORGANIZAR O RESULTADO
    # ========================================================

    df = (
        df
        .sort_values(CHAVES)
        .reset_index(drop=True)
    )

    return df

def verificar_chave_unica(df: pd.DataFrame, nome_view: str) -> None:
    """
    Verifica se existe mais de uma linha para a mesma combinação
    de id_property e reference_month.

    O processamento é interrompido caso existam duplicidades.
    """

    # Identificar todas as linhas que fazem parte de chaves duplicadas
    mascara_duplicadas = df.duplicated(subset=CHAVES, keep=False)

    df_duplicadas = (df.loc[mascara_duplicadas] .copy())

    if not df_duplicadas.empty:

        quantidade_linhas_duplicadas = len(
            df_duplicadas
        )

        quantidade_chaves_duplicadas = (
            df_duplicadas[CHAVES]
            .drop_duplicates()
            .shape[0]
        )

        exemplos = (
            df_duplicadas[CHAVES]
            .drop_duplicates()
            .sort_values(CHAVES)
            .head(10)
        )

        raise ValueError(
            f"\nA view {nome_view} possui duplicidades.\n"
            f"Linhas envolvidas: "
            f"{quantidade_linhas_duplicadas:,}\n"
            f"Chaves duplicadas: "
            f"{quantidade_chaves_duplicadas:,}\n\n"
            f"Exemplos:\n"
            f"{exemplos.to_string(index=False)}"
        )

    print(
        f"{nome_view}: chave única confirmada "
        f"em {len(df):,} linhas."
    )

##### Criar DataFrame Final

In [ ]:
# ============================================================
# PREPARAR TODAS AS VIEWS
# ============================================================

dados_preparados = {}

for nome_view, df_bruto in dados_views.items():

    print(f"\nPreparando {nome_view}...")

    df_preparado = preparar_view( df=df_bruto, nome_view=nome_view)
    
    verificar_chave_unica( df=df_preparado, nome_view=nome_view)

    dados_preparados[nome_view] = ( df_preparado.copy())

print("\nTodas as views foram preparadas.")

print("\nViews disponíveis em dados_preparados:")

for nome_view in dados_preparados:
    print(f"- {nome_view}")


Preparando vw_revenue...
vw_revenue: chave única confirmada em 10,541 linhas.

Preparando vw_asset_payment_history_monthly...
vw_asset_payment_history_monthly: chave única confirmada em 18,111 linhas.

Preparando vw_cattle...
vw_cattle: chave única confirmada em 10,578 linhas.

Preparando vw_expense...
vw_expense: chave única confirmada em 10,637 linhas.

Preparando vw_feeding...
vw_feeding: chave única confirmada em 10,052 linhas.

Preparando vw_labor...
vw_labor: chave única confirmada em 10,524 linhas.

Preparando vw_own_milk...
vw_own_milk: chave única confirmada em 8,884 linhas.

Preparando vw_area_ativa_mes...
vw_area_ativa_mes: chave única confirmada em 19,063 linhas.

Preparando vw_dairy_production_system_monthly...
vw_dairy_production_system_monthly: chave única confirmada em 11,731 linhas.

Todas as views foram preparadas.

Views disponíveis em dados_preparados:
- vw_revenue
- vw_asset_payment_history_monthly
- vw_cattle
- vw_expense
- vw_feeding
- vw_labor
- vw_own_milk
- v

In [ ]:
df_final = (
    dados_preparados[VIEW_BASE]
    .copy()
    .sort_values(CHAVES)
    .reset_index(drop=True)
)

quantidade_linhas_base = len(df_final)

print(f"\nBase revenue criada com " f"{quantidade_linhas_base:,} linhas.")

print(f"Propriedades: " f"{df_final['id_property'].nunique():,}")

print(f"Período: " f"{df_final['reference_month'].min():%Y-%m} " f"a {df_final['reference_month'].max():%Y-%m}")


Base revenue criada com 10,541 linhas.
Propriedades: 948
Período: 2025-01 a 2026-07


##### Realizar Merges

In [ ]:
# ============================================================
# MERGE DAS VIEWS COM A VW_REVENUE
# ============================================================

resumo_merge = []

for nome_view in VIEWS_MERGE:

    print(f"\nAdicionando {nome_view}...")

    # --------------------------------------------------------
    # 1. Conferir se a view foi preparada
    # --------------------------------------------------------

    if nome_view not in dados_preparados:
        raise KeyError(
            f"A view {nome_view} não foi encontrada "
            "em dados_preparados."
        )

    df_auxiliar = dados_preparados[nome_view].copy()

    # --------------------------------------------------------
    # 2. Conferir se as chaves existem
    # --------------------------------------------------------

    colunas_ausentes = [
        coluna
        for coluna in CHAVES
        if coluna not in df_auxiliar.columns
    ]

    if colunas_ausentes:
        raise KeyError(
            f"A view {nome_view} não possui as chaves "
            f"{colunas_ausentes}."
        )

    # --------------------------------------------------------
    # 3. Conferir novamente se a chave é única
    # --------------------------------------------------------

    duplicadas_auxiliar = df_auxiliar.duplicated(
        subset=CHAVES,
        keep=False,
    )

    if duplicadas_auxiliar.any():

        exemplos = (
            df_auxiliar.loc[
                duplicadas_auxiliar,
                CHAVES,
            ]
            .drop_duplicates()
            .sort_values(CHAVES)
            .head(10)
        )

        raise ValueError(
            f"A view {nome_view} possui duplicidades.\n\n"
            f"{exemplos.to_string(index=False)}"
        )

    # --------------------------------------------------------
    # 4. Renomear colunas que já existem no df_final
    # --------------------------------------------------------

    prefixo = nome_view.removeprefix("vw_")

    colunas_conflitantes = [
        coluna
        for coluna in df_auxiliar.columns
        if coluna not in CHAVES
        and coluna in df_final.columns
    ]

    if colunas_conflitantes:

        df_auxiliar = df_auxiliar.rename(
            columns={
                coluna: f"{prefixo}_{coluna}"
                for coluna in colunas_conflitantes
            }
        )

        print("Colunas renomeadas:", colunas_conflitantes)

    # --------------------------------------------------------
    # 5. Verificar a cobertura antes do merge
    # --------------------------------------------------------

    cobertura = (
        df_final[CHAVES]
        .merge(
            df_auxiliar[CHAVES],
            on=CHAVES,
            how="left",
            indicator=True,
            validate="one_to_one",
        )
    )

    chaves_encontradas = int(
        cobertura["_merge"]
        .eq("both")
        .sum()
    )

    chaves_sem_correspondencia = int(
        cobertura["_merge"]
        .eq("left_only")
        .sum()
    )

    linhas_antes = len(df_final)

    # --------------------------------------------------------
    # 6. Realizar o left merge
    # --------------------------------------------------------

    df_final = df_final.merge(
        df_auxiliar,
        on=CHAVES,
        how="left",
        validate="one_to_one",
    )

    linhas_depois = len(df_final)

    # --------------------------------------------------------
    # 7. Validar se a quantidade de linhas foi preservada
    # --------------------------------------------------------

    if linhas_antes != linhas_depois:
        raise ValueError(
            f"O merge com {nome_view} alterou a quantidade "
            f"de linhas de {linhas_antes:,} para "
            f"{linhas_depois:,}."
        )

    # --------------------------------------------------------
    # 8. Registrar o resumo
    # --------------------------------------------------------

    cobertura_percentual = round(
        chaves_encontradas / linhas_antes * 100,
        2,
    )

    resumo_merge.append({
        "view": nome_view,
        "linhas_view": len(df_auxiliar),
        "chaves_encontradas": chaves_encontradas,
        "chaves_sem_correspondencia": (
            chaves_sem_correspondencia
        ),
        "cobertura_percentual": cobertura_percentual,
        "colunas_adicionadas": (
            len(df_auxiliar.columns)
            - len(CHAVES)
        ),
        "linhas_antes": linhas_antes,
        "linhas_depois": linhas_depois,
    })

    print( f"Correspondências: " f"{chaves_encontradas:,}" )
    print( f"Sem correspondência: " f"{chaves_sem_correspondencia:,}" )
    print( f"Cobertura: " f"{cobertura_percentual:.2f}%" )
    print( f"Linhas antes/depois: " f"{linhas_antes:,} / {linhas_depois:,}" )


# ============================================================
# ORGANIZAR O RESULTADO FINAL
# ============================================================

df_final = (
    df_final
    .sort_values(CHAVES)
    .reset_index(drop=True)
)


# ============================================================
# VALIDAÇÕES FINAIS
# ============================================================

assert len(df_final) == quantidade_linhas_base, (
    "Os merges alteraram a quantidade de linhas da revenue."
)

assert not df_final.duplicated(CHAVES).any(), (
    "A tabela final possui duplicidades por "
    "id_property e reference_month."
)


# ============================================================
# RESUMO DOS MERGES
# ============================================================

df_resumo_merge = pd.DataFrame(
    resumo_merge
)

print("\nTodos os merges foram concluídos com sucesso.")
print(f"Linhas da base revenue: {quantidade_linhas_base:,}" )
print(f"Linhas da tabela final: {df_final.shape[0]:,}" )
print(f"Colunas da tabela final: {df_final.shape[1]:,}" )
print(f"Duplicidades por propriedade e mês: {df_final.duplicated(CHAVES).sum()}")

display(df_resumo_merge)
display(df_final.head())


Adicionando vw_asset_payment_history_monthly...
Correspondências: 10,297
Sem correspondência: 244
Cobertura: 97.69%
Linhas antes/depois: 10,541 / 10,541

Adicionando vw_cattle...
Correspondências: 10,207
Sem correspondência: 334
Cobertura: 96.83%
Linhas antes/depois: 10,541 / 10,541

Adicionando vw_expense...
Correspondências: 10,352
Sem correspondência: 189
Cobertura: 98.21%
Linhas antes/depois: 10,541 / 10,541

Adicionando vw_feeding...
Correspondências: 9,871
Sem correspondência: 670
Cobertura: 93.64%
Linhas antes/depois: 10,541 / 10,541

Adicionando vw_labor...
Correspondências: 10,301
Sem correspondência: 240
Cobertura: 97.72%
Linhas antes/depois: 10,541 / 10,541

Adicionando vw_own_milk...
Colunas renomeadas: ['hired_labor_quantity', 'family_labor_quantity']
Correspondências: 8,816
Sem correspondência: 1,725
Cobertura: 83.64%
Linhas antes/depois: 10,541 / 10,541

Adicionando vw_area_ativa_mes...
Correspondências: 10,405
Sem correspondência: 136
Cobertura: 98.71%
Linhas antes/dep

,view,linhas_view,chaves_encontradas,chaves_sem_correspondencia,cobertura_percentual,colunas_adicionadas,linhas_antes,linhas_depois
0,vw_asset_payment_history_monthly,18111,10297,244,97.69,4,10541,10541
1,vw_cattle,10578,10207,334,96.83,16,10541,10541
2,vw_expense,10637,10352,189,98.21,19,10541,10541
3,vw_feeding,10052,9871,670,93.64,9,10541,10541
4,vw_labor,10524,10301,240,97.72,4,10541,10541
5,vw_own_milk,8884,8816,1725,83.64,9,10541,10541
6,vw_area_ativa_mes,19063,10405,136,98.71,12,10541,10541
7,vw_dairy_production_system_monthly,11731,7627,2914,72.36,1,10541,10541


,id_property,reference_month,milk_sold_revenue,milk_volume_sold,milk_unit_price,ccs,cpp,fat,protein,received_loans,...,hectares_arrendada,hectares_app_reserva,hectares_atividade,hectares_area_total,land_value_propria,land_value_arrendada,land_value_app_reserva,land_value_atividade,land_value_area_total,production_system
0,004cde8f-4688-4b22-acd9-76151b7dcc7a,2025-04-01,112025.78,35677.0,3.14,421.0,77.0,3.48,3.40,0.0,...,25.18,1.18,24.0,25.18,NaN,0.0,0.0,0.0,0.0,UNSTRUCTURED_CONFINMENT
1,004cde8f-4688-4b22-acd9-76151b7dcc7a,2025-05-01,161358.96,54884.0,2.94,338.0,54.0,3.55,3.29,0.0,...,25.18,1.18,24.0,25.18,NaN,0.0,0.0,0.0,0.0,UNSTRUCTURED_CONFINMENT
2,004cde8f-4688-4b22-acd9-76151b7dcc7a,2025-06-01,157107.69,56311.0,2.79,192.0,102.0,3.46,3.32,0.0,...,25.18,1.18,24.0,25.18,NaN,0.0,0.0,0.0,0.0,UNSTRUCTURED_CONFINMENT
3,004cde8f-4688-4b22-acd9-76151b7dcc7a,2025-07-01,194327.56,69902.0,2.78,116.0,38.0,3.39,3.25,0.0,...,25.18,1.18,24.0,25.18,NaN,0.0,0.0,0.0,0.0,UNSTRUCTURED_CONFINMENT
4,004cde8f-4688-4b22-acd9-76151b7dcc7a,2025-08-01,194675.56,71836.0,2.71,132.0,41.0,3.47,3.24,0.0,...,25.18,1.18,24.0,25.18,NaN,0.0,0.0,0.0,0.0,UNSTRUCTURED_CONFINMENT


In [36]:
df_final

,id_property,reference_month,milk_sold_revenue,milk_volume_sold,milk_unit_price,ccs,cpp,fat,protein,received_loans,...,hectares_arrendada,hectares_app_reserva,hectares_atividade,hectares_area_total,land_value_propria,land_value_arrendada,land_value_app_reserva,land_value_atividade,land_value_area_total,production_system
0,004cde8f-4688-4b22-acd9-76151b7dcc7a,2025-04-01,112025.78,35677.0,3.14,421.0,77.0,3.48,3.40,0.0,...,25.18,1.18,24.00,25.18,NaN,0.0,0.0,0.000000,0.000000,UNSTRUCTURED_CONFINMENT
1,004cde8f-4688-4b22-acd9-76151b7dcc7a,2025-05-01,161358.96,54884.0,2.94,338.0,54.0,3.55,3.29,0.0,...,25.18,1.18,24.00,25.18,NaN,0.0,0.0,0.000000,0.000000,UNSTRUCTURED_CONFINMENT
2,004cde8f-4688-4b22-acd9-76151b7dcc7a,2025-06-01,157107.69,56311.0,2.79,192.0,102.0,3.46,3.32,0.0,...,25.18,1.18,24.00,25.18,NaN,0.0,0.0,0.000000,0.000000,UNSTRUCTURED_CONFINMENT
3,004cde8f-4688-4b22-acd9-76151b7dcc7a,2025-07-01,194327.56,69902.0,2.78,116.0,38.0,3.39,3.25,0.0,...,25.18,1.18,24.00,25.18,NaN,0.0,0.0,0.000000,0.000000,UNSTRUCTURED_CONFINMENT
4,004cde8f-4688-4b22-acd9-76151b7dcc7a,2025-08-01,194675.56,71836.0,2.71,132.0,41.0,3.47,3.24,0.0,...,25.18,1.18,24.00,25.18,NaN,0.0,0.0,0.000000,0.000000,UNSTRUCTURED_CONFINMENT
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10536,fff52e9f-83b6-44db-b7a4-01da7e77aeaa,2026-02-01,1897398.40,765080.0,2.48,170.0,18.0,3.79,3.31,0.0,...,0.00,59.44,207.54,266.98,153330.586561,NaN,80000.0,174332.658765,153330.586561,COMPOST_BARN
10537,fff52e9f-83b6-44db-b7a4-01da7e77aeaa,2026-03-01,2126922.40,765080.0,2.78,158.0,3.0,3.82,3.30,0.0,...,0.00,59.44,207.54,266.98,153330.586561,NaN,80000.0,174332.658765,153330.586561,COMPOST_BARN
10538,fff52e9f-83b6-44db-b7a4-01da7e77aeaa,2026-04-01,2311010.10,733654.0,3.15,167.0,9.0,4.04,3.33,0.0,...,0.00,59.44,207.54,266.98,153330.586561,NaN,80000.0,174332.658765,153330.586561,COMPOST_BARN
10539,fff52e9f-83b6-44db-b7a4-01da7e77aeaa,2026-05-01,2353731.83,751991.0,3.13,167.0,9.0,4.04,3.33,0.0,...,0.00,59.44,207.54,266.98,153330.586561,NaN,80000.0,174332.658765,153330.586561,COMPOST_BARN


##### Feature Engineering

In [ ]:
# ============================================================
# 1. PRESERVAR A BASE INTEGRADA
# ============================================================
df_integrada = df_final.copy()

CHAVES = ["id_property", "reference_month"]

linhas_iniciais = len(df_integrada)

print(f"Linhas: {df_integrada.shape[0]:,}")
print(f"Colunas antes das flags: {df_integrada.shape[1]:,}")


# ============================================================
# 2. MAPA DAS VIEWS E COLUNAS DE PRESENÇA
# ============================================================
MAPA_PRESENCA = {
    "vw_cattle": "has_cattle_data",
    "vw_expense": "has_expense_data",
    "vw_feeding": "has_feeding_data",
    "vw_labor": "has_labor_data",
    "vw_own_milk": "has_own_milk_data",
    "vw_asset_payment_history_monthly": "has_asset_data",
    "vw_area_ativa_mes": "has_active_area_data",
    "vw_dairy_production_system_monthly": "has_dairy_production_system_data",
}

# A revenue é a base da tabela final. Portanto, todas as linhas possuem revenue.
df_integrada["has_revenue_data"] = 1

# ============================================================
# 3. CRIAR AS FLAGS DE PRESENÇA
# ============================================================
for nome_view, nome_flag in MAPA_PRESENCA.items():

    print(f"Criando indicador de presença: {nome_flag}")

    if nome_view not in dados_preparados:
        raise KeyError( f"A view {nome_view} não foi encontrada em dados_preparados." )

    # Evita duplicação caso a célula seja executada novamente
    df_integrada = df_integrada.drop(
        columns=[nome_flag],
        errors="ignore",
    )

    # Uma linha por propriedade e mês presente na view
    chaves_disponiveis = (
        dados_preparados[nome_view][CHAVES]
        .dropna(subset=CHAVES)
        .drop_duplicates(subset=CHAVES)
        .assign(**{nome_flag: 1})
    )

    linhas_antes = len(df_integrada)

    df_integrada = df_integrada.merge(
        chaves_disponiveis,
        on=CHAVES,
        how="left",
        validate="one_to_one",
    )

    linhas_depois = len(df_integrada)

    if linhas_antes != linhas_depois:
        raise ValueError(
            f"O merge da flag {nome_flag} alterou a quantidade de linhas: "
            f"{linhas_antes:,} para {linhas_depois:,}."
        )

    df_integrada[nome_flag] = (
        df_integrada[nome_flag]
        .fillna(0)
        .astype("int8")
    )


# ============================================================
# 4. ORGANIZAR AS COLUNAS DE PRESENÇA
# ============================================================

colunas_presenca = [
    "has_revenue_data",
    "has_cattle_data",
    "has_expense_data",
    "has_feeding_data",
    "has_labor_data",
    "has_own_milk_data",
    "has_asset_data",
    "has_active_area_data",
    "has_dairy_production_system_data"
]


# ============================================================
# 5. VALIDAR O RESULTADO
# ============================================================

assert len(df_integrada) == linhas_iniciais, ("A criação das flags alterou a quantidade de linhas.")

assert not df_integrada.duplicated(CHAVES).any(), ("Foram geradas duplicidades por id_property e reference_month.")

print("\nFlags de presença criadas com sucesso.")
print(f"Linhas finais: {df_integrada.shape[0]:,}")
print(f"Colunas finais: {df_integrada.shape[1]:,}")


# ============================================================
# 6. CALCULAR A COBERTURA DE CADA VIEW
# ============================================================
df_cobertura_views = (
    df_integrada[colunas_presenca]
    .mean()
    .mul(100)
    .round(2)
    .rename("coverage_percentage")
    .rename_axis("source")
    .reset_index()
)

display(df_cobertura_views)

# ============================================================
# 7. CALCULAR INDICADORES PADRÃO
# ============================================================

# Renda do leite consumido
df_integrada['discarded_quantity_milk_revenue']   = df_integrada['discarded_quantity'] * df_integrada['milk_unit_price']
df_integrada['hired_labor_quantity_milk_revenue'] = df_integrada['own_milk_hired_labor_quantity'] * df_integrada['milk_unit_price']
df_integrada['discarded_quantity_milk_revenue']   = df_integrada['discarded_quantity'] * df_integrada['milk_unit_price']
df_integrada['family_labor_milk_revenue']         = df_integrada['own_milk_family_labor_quantity'] * df_integrada['milk_unit_price']
df_integrada['calves_quantity_milk_revenue']      = df_integrada['calves_quantity'] * df_integrada['milk_unit_price']

# Renda do leite
df_integrada['total_milk_revenue'] = df_integrada[[
    'milk_sold_revenue',
    'milk_derivatives_revenue',
    'price_bonus',
    'discarded_quantity_milk_revenue',
    'hired_labor_quantity_milk_revenue',
    'family_labor_milk_revenue',
    'calves_quantity_milk_revenue'
]].sum(axis=1).round(2) - df_integrada['price_penalty']

# Leite produzido
df_integrada['milk_produced'] = df_integrada[[
    'milk_volume_sold',
    'milk_volume_derivatives',
    'discarded_quantity',
    'own_milk_hired_labor_quantity',
    'own_milk_family_labor_quantity',
    'calves_quantity']].sum(axis=1).round(2)

# Renda da Atividade
df_integrada['total_activity_revenue'] = (
    df_integrada[[
        'total_milk_revenue',
        'received_loans',
        'animal_sale',
        'other_revenues',
        'voluminous_sold',
        'surplus_division',
    ]].sum(axis=1)
)
# Preço do Leite
df_integrada['milk_revenue_liter'] = df_integrada['total_milk_revenue'] / df_integrada['milk_produced']

# Leite diário
# Quantidade de dias do mês
df_integrada["days_in_month"] = ( df_integrada["reference_month"].dt.days_in_month)

# Produção média diária de leite
df_integrada["milk_daily"] = (df_integrada["milk_produced"] / df_integrada["days_in_month"].replace(0, np.nan))

# Vacas em lactação sobre o total de vacas
df_integrada["lactating_cows_total_cows"] = (df_integrada["lactating_cows"] / df_integrada["total_cows"].replace(0, np.nan)) * 100

# Vacas em lactação sobre o total do rebanho
df_integrada["lactating_cows_total_cattle"] = (df_integrada["lactating_cows"] / df_integrada["total_cattle"].replace(0, np.nan) ) * 100

# Ajustar mão de obra para dias/homem
df_integrada['hired_labor_quantity']  = df_integrada['hired_labor_quantity'] / df_integrada["days_in_month"]
df_integrada['family_labor_quantity'] = df_integrada['family_labor_quantity'] / df_integrada["days_in_month"]

# Quantidade total de mão de obra
df_integrada["total_labor_quantity"] = ( df_integrada[[ "hired_labor_quantity", "family_labor_quantity"]] .sum(axis=1, min_count=1) )

# Produção diária por unidade de mão de obra total
df_integrada["milk_total_labor_day"] = (df_integrada["milk_daily"] / df_integrada["total_labor_quantity"].replace(0, np.nan))

# Vacas em lactação por unidade de mão de obra
df_integrada["lactating_cows_total_labor"] = (df_integrada["lactating_cows"] / df_integrada["total_labor_quantity"].replace(0, np.nan))

# Custo de concentrado e minerais
df_integrada["concentrate_mineral_cost"] = (df_integrada["concentrate_amount_total"].fillna(0) + df_integrada["mineral_amount_total"].fillna(0))

# Custo total da alimentação
df_integrada["feeding_cost"] = (df_integrada["voluminous_amount_total"].fillna(0) + df_integrada["concentrate_amount_total"].fillna(0) + df_integrada["mineral_amount_total"].fillna(0) )

# Quantidade de concentrado 
df_integrada["concentrate_mineral_quantity"] = (df_integrada["concentrate_purchased_quantity"].fillna(0) + df_integrada["mineral_purchased_quantity"].fillna(0))

# Custo da alimentação por litro
df_integrada["feeding_cost_liter"] = (df_integrada["feeding_cost"] / df_integrada["milk_produced"].replace(0, np.nan))

# Custo de volumoso por litro
df_integrada["voluminous_cost_liter"] = (df_integrada["voluminous_amount_total"] / df_integrada["milk_produced"].replace(0, np.nan))

# Custo de concentrado e minerais por litro
df_integrada["concentrate_mineral_cost_liter"] = (df_integrada["concentrate_mineral_cost"] / df_integrada["milk_produced"].replace(0, np.nan))

# Participação do custo da alimentação no preço do leite
df_integrada["feeding_cost_milk_price"] = (df_integrada["feeding_cost_liter"] / df_integrada["milk_revenue_liter"].replace(0, np.nan) ) * 100

# Custo total da mão de obra
df_integrada["total_labor_expenses"] = (df_integrada[["hired_labor_expenses", "family_labor_expenses"]] .sum(axis=1, min_count=1))

# Custo da mão de obra contratada por litro
df_integrada["hired_labor_cost_liter"] = (df_integrada["hired_labor_expenses"] / df_integrada["milk_produced"].replace(0, np.nan))

# Custo da mão de obra familiar por litro
df_integrada["family_labor_cost_liter"] = (df_integrada["family_labor_expenses"] / df_integrada["milk_produced"].replace(0, np.nan))

# Custo total da mão de obra por litro
df_integrada["total_labor_cost_liter"] = (df_integrada["total_labor_expenses"] / df_integrada["milk_produced"].replace(0, np.nan))

# Participação da mão de obra na receita do leite
df_integrada["labor_cost_milk_revenue"] = ( df_integrada["total_labor_expenses"] / df_integrada["total_milk_revenue"].replace(0, np.nan) ) * 100


# ============================================================
# INDICADORES DE ÁREA
# ============================================================

# Vacas em lactação por hectare de atividade
df_integrada["lactating_cows_hectare_activity"] = (df_integrada["lactating_cows"] / df_integrada["hectares_atividade"].replace(0, np.nan))

# Percentual da área arrendada
df_integrada["rented_area_percentage"] = (df_integrada["hectares_arrendada"] / df_integrada["hectares_area_total"].replace(0, np.nan) * 100)

df_integrada.head(20)

Linhas: 10,541
Colunas antes das flags: 93
Criando indicador de presença: has_cattle_data
Criando indicador de presença: has_expense_data
Criando indicador de presença: has_feeding_data
Criando indicador de presença: has_labor_data
Criando indicador de presença: has_own_milk_data
Criando indicador de presença: has_asset_data
Criando indicador de presença: has_active_area_data
Criando indicador de presença: has_dairy_production_system_data

Flags de presença criadas com sucesso.
Linhas finais: 10,541
Colunas finais: 102


,source,coverage_percentage
0,has_revenue_data,100.00
1,has_cattle_data,96.83
2,has_expense_data,98.21
3,has_feeding_data,93.64
4,has_labor_data,97.72
5,has_own_milk_data,83.64
6,has_asset_data,97.69
7,has_active_area_data,98.71
8,has_dairy_production_system_data,72.36


,id_property,reference_month,milk_sold_revenue,milk_volume_sold,milk_unit_price,ccs,cpp,fat,protein,received_loans,...,labor_cost_milk_revenue,labor_cost_milk_revenue_percentage,milk_hectare_activity_day,milk_revenue_hectare_activity,lactating_cows_hectare_activity,total_cattle_hectare_activity,owned_area_percentage,rented_area_percentage,activity_area_percentage,app_reserve_area_percentage
0,004cde8f-4688-4b22-acd9-76151b7dcc7a,2025-04-01,112025.78,35677.00,3.1400,421.0,77.0,3.48,3.40,0.0,...,0.139584,13.958394,49.551389,4667.740833,3.500000,5.166667,0.000000,100.000000,95.313741,4.686259
1,004cde8f-4688-4b22-acd9-76151b7dcc7a,2025-05-01,161358.96,54884.00,2.9400,338.0,54.0,3.55,3.29,0.0,...,0.138210,13.820996,75.462366,6877.640000,3.250000,5.166667,0.000000,100.000000,95.313741,4.686259
2,004cde8f-4688-4b22-acd9-76151b7dcc7a,2025-06-01,157107.69,56311.00,2.7900,192.0,102.0,3.46,3.32,0.0,...,0.143134,14.313373,82.459722,6901.878750,3.375000,5.291667,0.000000,100.000000,95.313741,4.686259
3,004cde8f-4688-4b22-acd9-76151b7dcc7a,2025-07-01,194327.56,69902.00,2.7800,116.0,38.0,3.39,3.25,0.0,...,0.091903,9.190334,95.728495,8249.881667,3.500000,5.583333,0.000000,100.000000,95.313741,4.686259
4,004cde8f-4688-4b22-acd9-76151b7dcc7a,2025-08-01,194675.56,71836.00,2.7100,132.0,41.0,3.47,3.24,0.0,...,0.105250,10.525011,101.720430,8545.533333,3.375000,5.583333,0.000000,100.000000,95.313741,4.686259
5,004cde8f-4688-4b22-acd9-76151b7dcc7a,2025-09-01,184257.15,69531.00,2.6500,112.0,3.0,3.51,3.20,0.0,...,0.094620,9.461966,103.404167,8220.631250,3.541667,5.583333,0.000000,100.000000,95.313741,4.686259
6,004cde8f-4688-4b22-acd9-76151b7dcc7a,2025-10-01,174397.96,68932.00,2.5300,145.0,8.0,3.58,3.25,0.0,...,0.100509,10.050908,87.689367,6877.477037,2.962963,4.814815,0.000000,100.000000,95.812633,4.187367
7,004cde8f-4688-4b22-acd9-76151b7dcc7a,2025-11-01,141011.92,60781.00,2.3200,137.0,3.0,3.50,3.27,0.0,...,0.129088,12.908788,77.741975,5410.841481,2.592593,4.592593,0.000000,100.000000,95.812633,4.187367
8,004cde8f-4688-4b22-acd9-76151b7dcc7a,2025-12-01,115829.40,54380.00,2.1300,147.0,7.0,3.48,3.19,0.0,...,0.233101,23.310144,64.970131,4289.977778,3.851852,5.962963,0.000000,100.000000,95.812633,4.187367
9,004cde8f-4688-4b22-acd9-76151b7dcc7a,2026-01-01,140414.56,67184.00,2.0900,250.0,8.0,3.44,3.29,0.0,...,0.137944,13.794439,88.156511,5711.660370,3.777778,5.666667,0.000000,100.000000,95.812633,4.187367


In [26]:
df_integrada.columns.tolist()

['id_property',
 'reference_month',
 'milk_sold_revenue',
 'milk_volume_sold',
 'milk_unit_price',
 'ccs',
 'cpp',
 'fat',
 'protein',
 'received_loans',
 'animal_sale',
 'other_revenues',
 'price_bonus',
 'price_penalty',
 'milk_derivatives_revenue',
 'milk_volume_derivatives',
 'unit_price_derivative',
 'voluminous_sold',
 'surplus_division',
 'monthly_depreciation_benfeitorias',
 'monthly_depreciation_maquinas_e_equipamentos',
 'monthly_average_capital_stock_benfeitorias',
 'monthly_average_capital_stock_maquinas_e_equipamentos',
 'lactating_cows',
 'dry_cows',
 'nursing',
 'rearing',
 'males',
 'other_categories',
 'total_cows',
 'total_cattle',
 'lactating_cows_value',
 'dry_cows_value',
 'nursing_value',
 'rearing_value',
 'males_value',
 'other_categories_value',
 'total_cows_value',
 'total_cattle_value',
 'general_expenses',
 'advance_payment',
 'administration',
 'land_lease',
 'technical_assistance',
 'animal_purchase',
 'land_purchase',
 'repairs',
 'loan_interest',
 'hormo

#### Importar Dimensão Produtor

In [ ]:
from sqlalchemy import text

# ============================================================
# IMPORTAR APENAS AS COLUNAS DIMENSIONAIS NECESSÁRIAS
# ============================================================

consulta_dim_property = text(
    """
    SELECT
        id_property,
        property_name,
        labor_rural_code,
        entrepreneur_name,
        agroindustry_name,
        dairy_region,
        property_status
    FROM analytics_int.vw_dim_property_base;
    """
)

df_dim_property = pd.read_sql_query( consulta_dim_property, con=engine, )

# Padronizar a chave
df_dim_property["id_property"] = ( df_dim_property["id_property"] .astype("string") .str.strip() )

# Remover linhas sem chave
df_dim_property = ( df_dim_property .dropna(subset=["id_property"]) .reset_index(drop=True) )

# Validar uma linha por propriedade
if df_dim_property.duplicated("id_property").any():
    raise ValueError( "A dimensão possui mais de uma linha por id_property." )

# Evitar conflito com property_name já existente
df_integrada = df_integrada.drop( columns=["property_name"], errors="ignore", )

# Merge dimensional
linhas_antes = len(df_integrada)

df_integrada = df_integrada.merge(
    df_dim_property,
    on="id_property",
    how="left",
    validate="many_to_one",
)

if len(df_integrada) != linhas_antes:
    raise ValueError( "O merge dimensional alterou a quantidade de linhas." )

print("Merge dimensional concluído.")
print(f"Linhas: {df_integrada.shape[0]:,}")
print(f"Colunas: {df_integrada.shape[1]:,}")

df_integrada.head()

Merge dimensional concluído.
Linhas: 10,541
Colunas: 144


,id_property,reference_month,milk_sold_revenue,milk_volume_sold,milk_unit_price,ccs,cpp,fat,protein,received_loans,...,owned_area_percentage,rented_area_percentage,activity_area_percentage,app_reserve_area_percentage,property_name,labor_rural_code,entrepreneur_name,agroindustry_name,dairy_region,property_status
0,004cde8f-4688-4b22-acd9-76151b7dcc7a,2025-04-01,112025.78,35677.0,3.14,421.0,77.0,3.48,3.40,0.0,...,0.0,100.0,95.313741,4.686259,Fazenda São Bartolomeu,LR01964,Edgar Jose de Azevedo Neto,NESTLÉ,Ibiá - 1215,active_approved
1,004cde8f-4688-4b22-acd9-76151b7dcc7a,2025-05-01,161358.96,54884.0,2.94,338.0,54.0,3.55,3.29,0.0,...,0.0,100.0,95.313741,4.686259,Fazenda São Bartolomeu,LR01964,Edgar Jose de Azevedo Neto,NESTLÉ,Ibiá - 1215,active_approved
2,004cde8f-4688-4b22-acd9-76151b7dcc7a,2025-06-01,157107.69,56311.0,2.79,192.0,102.0,3.46,3.32,0.0,...,0.0,100.0,95.313741,4.686259,Fazenda São Bartolomeu,LR01964,Edgar Jose de Azevedo Neto,NESTLÉ,Ibiá - 1215,active_approved
3,004cde8f-4688-4b22-acd9-76151b7dcc7a,2025-07-01,194327.56,69902.0,2.78,116.0,38.0,3.39,3.25,0.0,...,0.0,100.0,95.313741,4.686259,Fazenda São Bartolomeu,LR01964,Edgar Jose de Azevedo Neto,NESTLÉ,Ibiá - 1215,active_approved
4,004cde8f-4688-4b22-acd9-76151b7dcc7a,2025-08-01,194675.56,71836.0,2.71,132.0,41.0,3.47,3.24,0.0,...,0.0,100.0,95.313741,4.686259,Fazenda São Bartolomeu,LR01964,Edgar Jose de Azevedo Neto,NESTLÉ,Ibiá - 1215,active_approved


#### Preparar os dados para o Excel

In [ ]:
def preparar_dataframe_para_excel(df: pd.DataFrame) -> pd.DataFrame:
    """
    Prepara um DataFrame para exportação pelo openpyxl.
    """

    df_excel = df.copy()

    # Excel não trabalha com infinito
    df_excel = df_excel.replace([np.inf, -np.inf], np.nan)

    # Excel não aceita datas com timezone
    for coluna in df_excel.columns:
        if isinstance( df_excel[coluna].dtype, pd.DatetimeTZDtype, ):
            df_excel[coluna] = ( df_excel[coluna] .dt.tz_localize(None) )

    return df_excel

#### Exportar uma planilha por view

In [ ]:
# ============================================================
# EXPORTAR DF_INTEGRADA PARA EXCEL
# ============================================================

DATA_EXPORTACAO = datetime.now().strftime("%Y_%m_%d")
PASTA_SAIDA = (Path.cwd().parent / "data" / "outputs" )
PASTA_SAIDA.mkdir(parents=True, exist_ok=True)
CAMINHO_ARQUIVO = (PASTA_SAIDA / f"{DATA_EXPORTACAO}_indicadores_mensais.xlsx")

# Criar uma cópia para não alterar a tabela original
df_exportacao = df_integrada.copy()

# Remover valores infinitos
df_exportacao = df_exportacao.replace([np.inf, -np.inf], np.nan)

# Ordenar a tabela
df_exportacao = (df_exportacao.sort_values( ["id_property", "reference_month"] ) .reset_index(drop=True))

# Validar duplicidades
if df_exportacao.duplicated(["id_property", "reference_month"]).any():
    raise ValueError("Existem duplicidades por id_property e reference_month.")

# Preparar os dados para o Excel
df_exportacao = preparar_dataframe_para_excel(df_exportacao)

# Exportar com a formatação padrão
exportar_varias_abas_xlsx(abas={"Indicadores Mensais": df_exportacao }, caminho_saida=CAMINHO_ARQUIVO, fonte="Aptos")

print("Exportação concluída com sucesso.")
print(f"Linhas exportadas: {len(df_exportacao):,}")
print(f"Arquivo: {CAMINHO_ARQUIVO}")

Exportação concluída com sucesso.
Linhas exportadas: 10,541
Arquivo: c:\Users\Guilherme\LABOR RURAL\Analytics - Departamento Analytics\TEMP\GUILHERME\SCRIPTS\projetcs\elabore-views\data\outputs\2026_07_16_indicadores_mensais.xlsx


## Regras de conssitência dos Indicadores Mensais